In [2]:
import os
import shutil
from pathlib import Path

# Define paths
WORK_DIR = Path('/kaggle/working')
DATA_DIR = WORK_DIR / 'data'
MODELS_DIR = WORK_DIR / 'models'
SCRIPTS_DIR = WORK_DIR / 'scripts'

# Create directories
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
SCRIPTS_DIR.mkdir(exist_ok=True)
(WORK_DIR / 'outputs' / 'checkpoints').mkdir(parents=True, exist_ok=True)

# 1. Create __init__.py files so Python treats folders as modules
(MODELS_DIR / '__init__.py').touch()
(DATA_DIR / '__init__.py').touch()

# 2. Copy Python Scripts from Kaggle inputs to working directory
shutil.copy('/kaggle/input/datasets/anybodyk/custom-ds/custom_dataset.py', DATA_DIR / 'custom_dataset.py')
shutil.copy('/kaggle/input/datasets/anybodyk/models/fusion_network.py', MODELS_DIR / 'fusion_network.py')
shutil.copy('/kaggle/input/datasets/anybodyk/models/interpretability.py', MODELS_DIR / 'interpretability.py')
shutil.copy('/kaggle/input/datasets/anybodyk/scriptss/train.py', SCRIPTS_DIR / 'train.py')
shutil.copy('/kaggle/input/datasets/anybodyk/scriptss/evaluate.py', SCRIPTS_DIR / 'evaluate.py')

# 3. Copy CSV
shutil.copy('/kaggle/input/datasets/anybodyk/verified-live-vidss/verified_live_videos.csv', DATA_DIR / 'verified_live_videos.csv')

print("✅ Workspace assembled successfully!")

✅ Workspace assembled successfully!


In [3]:
import yaml
import glob

# Find the extracted_frames directory (Kaggle unzips can be deeply nested)
frame_dirs = glob.glob('/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames', recursive=True)

if not frame_dirs:
    print("❌ ERROR: Could not find extracted_frames. Please check the dataset structure.")
else:
    FRAMES_PATH = frame_dirs[0]
    print(f"✅ Found frames at: {FRAMES_PATH}")

    # Copy config to working directory
    shutil.copy('/kaggle/input/datasets/anybodyk/configg/config.yaml', WORK_DIR / 'config.yaml')
    
    # Read and update the config
    with open(WORK_DIR / 'config.yaml', 'r') as f:
        cfg = yaml.safe_load(f)
        
    # Overwrite local PC paths with Kaggle paths
    cfg['data']['frames_dir'] = FRAMES_PATH
    cfg['data']['verified_csv'] = str(DATA_DIR / 'verified_live_videos.csv')
    
    # Save the updated config
    with open(WORK_DIR / 'config.yaml', 'w') as f:
        yaml.dump(cfg, f)
        
    print("✅ Config updated successfully! Ready for training.")

✅ Found frames at: /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
✅ Config updated successfully! Ready for training.


In [5]:
import os
os.chdir("/kaggle/working")
print("CWD:", os.getcwd())
print("config exists:", os.path.exists("config.yaml"))
print("frames_dir:", __import__("yaml").safe_load(open("config.yaml"))["data"]["frames_dir"])

CWD: /kaggle/working
config exists: True
frames_dir: /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames


In [7]:
import os, glob, yaml
from pathlib import Path

# Find extracted_frames anywhere under /kaggle/input
candidates = glob.glob("/kaggle/input/**/extracted_frames", recursive=True)
print("Candidates:", candidates)

for c in candidates:
    subdirs = list(Path(c).iterdir())
    print(f"\n{c}")
    print(f"  video folders: {len(subdirs)}")
    if subdirs:
        sample = subdirs[0]
        print(f"  sample: {sample.name} -> {list(sample.glob('*.png'))}")

# Show what config currently has
with open("/kaggle/working/config.yaml") as f:
    cfg = yaml.safe_load(f)
print("\nCurrent frames_dir:", cfg["data"]["frames_dir"])
print("Exists?", os.path.isdir(cfg["data"]["frames_dir"]))

Candidates: ['/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames']

/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
  video folders: 8047
  sample: moILQq33PkM -> [PosixPath('/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames/moILQq33PkM/frame_2.png'), PosixPath('/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames/moILQq33PkM/frame_1.png'), PosixPath('/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames/moILQq33PkM/frame_0.png')]

Current frames_dir: /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
Exists? True


In [9]:
import glob, yaml, os
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

# Pick the extracted_frames folder that actually has video subfolders
best = None
for c in glob.glob("/kaggle/input/**/extracted_frames", recursive=True):
    p = Path(c)
    if any(p.iterdir()):
        best = str(p)
        break

if not best:
    raise RuntimeError("No extracted_frames found! Check your Kaggle dataset upload.")

print("Using frames_dir:", best)
print("Folder count:", len(list(Path(best).iterdir())))

with open(WORK_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["frames_dir"] = best
cfg["data"]["verified_csv"] = "/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_data_meta/data/verified_live_videos.csv"
# Or whichever path has your CSV — use the one that exists:
# cfg["data"]["verified_csv"] = "/kaggle/input/datasets/anybodyk/verified_live_vidss/verified_live_videos.csv"

with open(WORK_DIR / "config.yaml", "w") as f:
    yaml.dump(cfg, f)

# Quick sanity: one real video from CSV
import pandas as pd
df = pd.read_csv(cfg["data"]["verified_csv"])
vid = str(df.iloc[0]["video_id"])
frame_path = Path(best) / vid / "frame_0.png"
print("Sample frame exists:", frame_path.exists(), frame_path)

Using frames_dir: /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
Folder count: 8047
Sample frame exists: False /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames/J9xErXLh3bo/frame_0.png


In [11]:
import os
from pathlib import Path

src = Path("/kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames")
dst = Path("/kaggle/working/data/extracted_frames")

dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists() or dst.is_symlink():
    print("Link already exists:", dst)
else:
    os.symlink(src, dst)
    print("Symlinked:", dst, "->", src)

# verify
print("HPa6mRwjUg8 frame exists:", (dst / "HPa6mRwjUg8" / "frame_0.png").exists())

Symlinked: /kaggle/working/data/extracted_frames -> /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
HPa6mRwjUg8 frame exists: True


In [12]:
import os
os.chdir("/kaggle/working")
!python scripts/train.py --config config.yaml --ablation text_only

Using device: cuda
2026-06-29 14:03:47,686 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 14:03:47,702 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 14:03:47,763 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 14:03:47,830 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 14:03:47,830 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-29 14:03:47,922 | INFO | HTTP Request: GET https://huggingface.co/api/models/sagorsarker/bangla-bert-base/tr

In [13]:
!python scripts/train.py --config config.yaml --ablation vision_only

Using device: cuda
2026-06-29 14:42:20,529 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 14:42:20,530 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-29 14:42:20,546 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 14:42:20,616 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 14:42:20,682 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 14:42:20,746 | INFO | HTTP Request: GET https://huggingface.co/api/models/sagorsarker/bangla-bert-base/tr

In [14]:
import os
from pathlib import Path

ckpt = Path("/kaggle/working/outputs/checkpoints")

# Safe to delete — only used for --resume, not evaluation
for name in [
    "last_model_text_only.pt",
    "last_model_vision_only.pt",
]:
    p = ckpt / name
    if p.exists():
        size_gb = p.stat().st_size / 1e9
        p.unlink()
        print(f"Deleted {name} ({size_gb:.2f} GB)")
    else:
        print(f"Not found: {name}")

# Confirm what remains
print("\nRemaining checkpoints:")
for p in sorted(ckpt.glob("*.pt")):
    print(f"  {p.name}: {p.stat().st_size / 1e9:.2f} GB")

Deleted last_model_text_only.pt (3.03 GB)
Deleted last_model_vision_only.pt (3.03 GB)

Remaining checkpoints:
  best_model_text_only.pt: 3.03 GB
  best_model_vision_only.pt: 3.03 GB


In [15]:
!python scripts/train.py --config config.yaml --ablation full  

Using device: cuda
2026-06-29 15:24:23,837 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 15:24:23,853 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 15:24:23,917 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 15:24:23,918 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-29 15:24:23,984 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 15:24:24,047 | INFO | HTTP Request: GET https://huggingface.co/api/models/sagorsarker/bangla-bert-base/tr

In [18]:
from pathlib import Path

eval_path = Path("/kaggle/working/scripts/evaluate.py")
text = eval_path.read_text()

old = '''    resolved_path = resolve_checkpoint_path(checkpoint_path, condition=condition)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model'''

new = '''    resolved_path = resolve_checkpoint_path(checkpoint_path, condition=condition)
    checkpoint = torch.load(resolved_path, map_location=device)
    model = VTCF(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    logger.info("Loaded checkpoint from %s", resolved_path)
    return model'''

if old not in text:
    raise RuntimeError("evaluate.py already patched or differs — check file manually")

eval_path.write_text(text.replace(old, new))
print("✅ evaluate.py patched")

✅ evaluate.py patched


In [19]:
import os
os.chdir("/kaggle/working")
!python scripts/evaluate.py --config config.yaml

Using device: cuda
2026-06-29 16:09:36,254 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 16:09:36,269 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 16:09:36,338 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 16:09:36,339 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-29 16:09:36,403 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 16:09:36,464 | INFO | HTTP Request: GET https://huggingface.co/api/models/sagorsarker/bangla-bert-base/tr

In [20]:
from pathlib import Path

ckpt = Path("/kaggle/working/outputs/checkpoints")
print("Exists:", ckpt.exists())
if ckpt.exists():
    for p in sorted(ckpt.glob("*.pt")):
        print(p.name, f"({p.stat().st_size / 1e9:.2f} GB)")
else:
    print("No checkpoints folder found.")

Exists: True
best_model_full.pt (3.03 GB)
best_model_text_only.pt (3.03 GB)
best_model_vision_only.pt (3.03 GB)
last_model_full.pt (3.03 GB)


In [21]:
!zip -r /kaggle/working/vtcf_checkpoints.zip /kaggle/working/outputs/checkpoints/

  adding: kaggle/working/outputs/checkpoints/ (stored 0%)
  adding: kaggle/working/outputs/checkpoints/best_model_vision_only.pt (deflated 34%)
  adding: kaggle/working/outputs/checkpoints/best_model_full.pt (deflated 25%)
  adding: kaggle/working/outputs/checkpoints/best_model_text_only.pt (deflated 48%)
  adding: kaggle/working/outputs/checkpoints/last_model_full.pt (deflated 25%)


In [22]:
from pathlib import Path
p = Path("/kaggle/working/vtcf_checkpoints.zip")
print(f"Size: {p.stat().st_size / 1e9:.2f} GB")
print("Complete:", p.exists())

Size: 8.14 GB
Complete: True


In [23]:
!cd /kaggle/working && split -b 1500M vtcf_checkpoints.zip vtcf_part_
!ls -lh /kaggle/working/vtcf_part_*

split: vtcf_part_aa: No space left on device
-rw-r--r-- 1 root root 636M Jun 29 16:38 /kaggle/working/vtcf_part_aa


In [24]:
!du -sh /kaggle/working/* /kaggle/working/outputs/* 2>/dev/null
!ls -lh /kaggle/working/outputs/checkpoints/ 2>/dev/null || echo "checkpoints folder gone"
!ls -lh /kaggle/working/vtcf*

4.0K	/kaggle/working/config.yaml
2.8M	/kaggle/working/data
104K	/kaggle/working/models
12G	/kaggle/working/outputs
60K	/kaggle/working/scripts
7.6G	/kaggle/working/vtcf_checkpoints.zip
636M	/kaggle/working/vtcf_part_aa
total 12G
-rw-r--r-- 1 root root 2.9G Jun 29 15:33 best_model_full.pt
-rw-r--r-- 1 root root 2.9G Jun 29 14:17 best_model_text_only.pt
-rw-r--r-- 1 root root 2.9G Jun 29 14:58 best_model_vision_only.pt
-rw-r--r-- 1 root root 2.9G Jun 29 16:00 last_model_full.pt
-rw-r--r-- 1 root root 7.6G Jun 29 16:32 /kaggle/working/vtcf_checkpoints.zip
-rw-r--r-- 1 root root 636M Jun 29 16:38 /kaggle/working/vtcf_part_aa


In [25]:
from pathlib import Path

# Failed split chunk
Path("/kaggle/working/vtcf_part_aa").unlink(missing_ok=True)

# Original .pt files — already inside the zip
for p in Path("/kaggle/working/outputs/checkpoints").glob("*.pt"):
    print(f"Deleting {p.name}")
    p.unlink()

!du -sh /kaggle/working/*
!df -h /kaggle/working

Deleting best_model_vision_only.pt
Deleting best_model_full.pt
Deleting best_model_text_only.pt
Deleting last_model_full.pt
4.0K	/kaggle/working/config.yaml
2.8M	/kaggle/working/data
104K	/kaggle/working/models
28K	/kaggle/working/outputs
60K	/kaggle/working/scripts
7.6G	/kaggle/working/vtcf_checkpoints.zip
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  7.6G   12G  39% /kaggle/working


In [26]:
!cd /kaggle/working && split -b 1500M vtcf_checkpoints.zip vtcf_part_
!ls -lh /kaggle/working/vtcf_part_*

-rw-r--r-- 1 root root 1.5G Jun 29 16:42 /kaggle/working/vtcf_part_aa
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ab
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ac
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ad
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ae
-rw-r--r-- 1 root root 260M Jun 29 16:43 /kaggle/working/vtcf_part_af


In [27]:
from pathlib import Path
p = Path("/kaggle/working/vtcf_checkpoints.zip")
if p.exists():
    p.unlink()
    print("Deleted zip — parts kept")
!df -h /kaggle/working
!ls -lh /kaggle/working/vtcf_part_*

Deleted zip — parts kept
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   16G  4.4G  78% /kaggle/working
-rw-r--r-- 1 root root 1.5G Jun 29 16:42 /kaggle/working/vtcf_part_aa
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ab
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ac
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ad
-rw-r--r-- 1 root root 1.5G Jun 29 16:43 /kaggle/working/vtcf_part_ae
-rw-r--r-- 1 root root 260M Jun 29 16:43 /kaggle/working/vtcf_part_af
